In [1]:
import pandas as pd
candy = pd.read_csv('../data/candy-data.csv')

In [2]:
# theoretically, the int features could be converted to booleans, however this would make using ols more difficult later on, so we will leave them as is

In [3]:
candy['winpercent'] = candy['winpercent'].apply(lambda x: round(x / 100, 3))

In [4]:
#we do not standardize the numeric features, they are already derived variables so we will leave them as it is, winpercent is target variable and as such will not be standardized

The dataset is quite limited in terms of number of features and their types (mostly binary). Statistically, adding more feature will not help us build more reliable model as also number of observations (candy types) is not that high, however we can at least get a bit better idea of what else might influence the candy popularity. One big latent (hidden) feature might be brand popularity. Let's match the product with it brand using llm. We will also manually check for some major problems, but generally will believe the output of ChatGPT.

In [ ]:
from tqdm import tqdm
from utils import get_gpt_response
prompt = 'Determine mother company of the following candy brand. Return only the mother company name. If unknown, return "Unknown".\n\n Candy brand:'
companies = []
for name in tqdm(candy['competitorname']):
    competitor_name = f' {name}'
    response = get_gpt_response(prompt + competitor_name)
    companies.append(response)
# originally, we did this in bulk (all the names at once), but in that case, we had a problem of stability of the output (sometimes the number of returned companies did not match the number of candy brands)

In [7]:
#to check if the number of companies matches the number of candy brands
print(len(companies))

85


In [9]:
candy['mother_company'] = [company.strip() for company in companies]
candy[['competitorname', 'mother_company']]

,competitorname,mother_company
0,100 Grand,Ferrara Candy Company
1,3 Musketeers,"Mars, Incorporated"
2,One dime,Unknown
3,One quarter,Unknown
4,Air Heads,Perfetti Van Melle
...,...,...
80,Twizzlers,The Hershey Company
81,Warheads,Impact Confections
82,Welch's Fruit Snacks,"The Promotion In Motion Companies, Inc."
83,Werther's Original Caramel,August Storck KG


In [34]:
candy.mother_company.value_counts()

mother_company
Hershey's                                  16
Ferrero SpA                                15
Mars, Incorporated                         13
Tootsie Roll Industries                    12
Nestlé                                      6
Mondelez International                      4
Haribo                                      4
Unknown                                     4
Perfetti Van Melle                          1
Spangler Candy Company                      1
Just Born                                   1
Zeta Espacial S.A.                          1
American Licorice Company                   1
Topps                                       1
SweetWorks Confections                      1
Mars, Inc.                                  1
Impact Confections                          1
The Promotion In Motion Companies, Inc.     1
August Storck KG                            1
Name: count, dtype: int64

In [32]:
candy.loc[candy['mother_company'] == 'Ferrero SpA', ['competitorname', 'mother_company']]

,competitorname,mother_company
6,Baby Ruth,Ferrero SpA
17,Gobstopper,Ferrero SpA


In [65]:
candy = candy.replace("The Hershey Company", "Hershey's")
candy = candy.replace("Nestlé S.A.", "Nestlé")
candy = candy.replace("Mars, Incorporated", "Mars, Inc.")
candy = candy.replace("Haribo GmbH & Co. KG", "Haribo")
candy = candy.replace("Ferrara Candy Company", "Ferrero SpA")

In [35]:
candy[['competitorname', 'mother_company']]

,competitorname,mother_company
0,100 Grand,Ferrero SpA
1,3 Musketeers,"Mars, Incorporated"
2,One dime,Unknown
3,One quarter,Unknown
4,Air Heads,Perfetti Van Melle
...,...,...
80,Twizzlers,Hershey's
81,Warheads,Impact Confections
82,Welch's Fruit Snacks,"The Promotion In Motion Companies, Inc."
83,Werther's Original Caramel,August Storck KG


In [ ]:
candy.mother_company.value_counts()

In [67]:
big_brands = candy.mother_company.value_counts().index[:7]

In [68]:
candy['big_brand'] = candy.mother_company.apply(lambda x: 1 if x in big_brands else 0)

In [69]:
candy.to_csv('/Users/A107809368/projects/playground/candy_case_study/data/candy-data-cleaned.csv', index=False)

In [2]:
import pandas as pd
candy = pd.read_csv('/Users/A107809368/projects/playground/candy_case_study/data/candy-data-cleaned.csv')

In [5]:
candy.mother_company.value_counts()

mother_company
Hershey's                                  16
Ferrero SpA                                15
Mars, Inc.                                 14
Tootsie Roll Industries                    12
Nestlé                                      6
Mondelez International                      4
Haribo                                      4
Unknown                                     4
Perfetti Van Melle                          1
Spangler Candy Company                      1
Just Born                                   1
Zeta Espacial S.A.                          1
American Licorice Company                   1
Topps                                       1
SweetWorks Confections                      1
Impact Confections                          1
The Promotion In Motion Companies, Inc.     1
August Storck KG                            1
Name: count, dtype: int64